In [1]:
import os
import math
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, datasets
import timm
from PIL import Image
import random
from tqdm.auto import tqdm

# Mixed precision
from torch.cuda.amp import autocast, GradScaler

/opt/anaconda3/envs/GoAI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = r"/path/to/your/data"   # ImageFolder structure: DATA_DIR/train/<class>/*.jpg etc.
NUM_CLASSES = 60
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32          # adjust to fit T4 (try 32, 24, 16)
IMAGE_INPUT = 224        # final input size to model
PATCH_SIZE = 300         # crop size from original then resize to 224 (you used 300 -> 224)
PATCHES_PER_IMAGE = 8    # how many random patches per image during training
EPOCHS = 40
MODEL_NAME = "efficientnetv2_s"  # change to "convnext_tiny", "swin_tiny_patch4_window7_224" to test
OUT_DIR = "./checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
train_transform = transforms.Compose([
    transforms.Resize(320),                     # for slight scale variance
    transforms.RandomResizedCrop(PATCH_SIZE, scale=(0.8,1.0), ratio=(0.9,1.1)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.08, contrast=0.12, saturation=0.06),
    transforms.RandomErasing(p=0.2, scale=(0.02,0.2), ratio=(0.3,3.3)),
    transforms.CenterCrop(PATCH_SIZE),
    transforms.Resize(IMAGE_INPUT),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(PATCH_SIZE),
    transforms.Resize(IMAGE_INPUT),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
class PatchImageFolder(Dataset):
    """
    Wrap ImageFolder: each item returns one random patch (PATCH_SIZE) from the original image.
    This avoids storing patches on disk. For validation we still sample deterministic center crop.
    """
    def __init__(self, root, transform, patches_per_image=1, is_train=True):
        self.dataset = datasets.ImageFolder(root)
        self.transform = transform
        self.patches_per_image = patches_per_image
        self.is_train = is_train
        # store paths and labels
        self.samples = self.dataset.samples  # list of (path, class_idx)

    def __len__(self):
        # if training and we request multiple patches per image, increase effective length
        return len(self.samples) * (self.patches_per_image if self.is_train else 1)

    def __getitem__(self, idx):
        # map idx to image index
        img_idx = idx // (self.patches_per_image if self.is_train else 1)
        path, label = self.samples[img_idx]
        img = Image.open(path).convert("RGB")
        if self.is_train:
            # random crop region with some texture heuristic (simple random)
            # You could refine: sample more from high-edge regions using cv2 if needed
            # We'll use the transform pipeline that includes RandomResizedCrop
            sample = self.transform(img)
        else:
            sample = self.transform(img)
        return sample, label

In [5]:
train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")

# create datasets
train_dataset = PatchImageFolder(train_dir, transform=train_transform,
                                 patches_per_image=PATCHES_PER_IMAGE, is_train=True)
val_dataset = PatchImageFolder(val_dir, transform=val_transform, patches_per_image=1, is_train=False)

# compute class weights for sampler and loss
# count samples per class in original dataset (not per-patch)
orig_train = datasets.ImageFolder(train_dir)
counts = Counter([lab for (_, lab) in orig_train.samples])
class_counts = np.array([counts[i] for i in range(len(orig_train.classes))])
# avoid zero
class_counts = np.maximum(class_counts, 1)
class_weights = 1.0 / class_counts
# per-sample weights:
sample_weights = [class_weights[label] for (_, label) in orig_train.samples]
# expand sample_weights to match patches_per_image in train_dataset length
sample_weights_expanded = np.repeat(np.array(sample_weights), PATCHES_PER_IMAGE)
sampler = WeightedRandomSampler(weights=sample_weights_expanded,
                                num_samples=len(sample_weights_expanded),
                                replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

print(f"Classes: {len(orig_train.classes)}, train images: {len(orig_train)}, "
      f"effective train samples (patches): {len(train_dataset)}")

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/your/data/train'

In [ ]:
def create_model(model_name=MODEL_NAME, num_classes=NUM_CLASSES, pretrained=True):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

model = create_model(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
# scheduler: cosine annealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
scaler = GradScaler()
best_val_acc = 0.0
patience = 6
stopping_counter = 0

def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    pbar = tqdm(train_loader, desc=f"Train E{epoch}")
    for xb, yb in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with autocast():
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        preds = torch.argmax(logits, dim=1)
        running_loss += loss.item() * xb.size(0)
        running_corrects += (preds == yb).sum().item()
        total += xb.size(0)
        pbar.set_postfix(loss=running_loss/total, acc=running_corrects/total)
    epoch_loss = running_loss / total
    epoch_acc = running_corrects / total
    return epoch_loss, epoch_acc

def validate(epoch):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Val E{epoch}")
        for xb, yb in pbar:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(xb)
                loss = criterion(logits, yb)
            preds = torch.argmax(logits, dim=1)
            running_loss += loss.item() * xb.size(0)
            running_corrects += (preds == yb).sum().item()
            total += xb.size(0)
            pbar.set_postfix(loss=running_loss/total, acc=running_corrects/total)
    epoch_loss = running_loss / total
    epoch_acc = running_corrects / total
    return epoch_loss, epoch_acc

for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = train_one_epoch(epoch)
    val_loss, val_acc = validate(epoch)
    scheduler.step()
    print(f"Epoch {epoch} Train loss {train_loss:.4f} acc {train_acc:.4f} | Val loss {val_loss:.4f} acc {val_acc:.4f}")

    # checkpoint by best val acc
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        stopping_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, os.path.join(OUT_DIR, f"best_{MODEL_NAME}_acc{val_acc:.4f}.pth"))
        print("Saved best checkpoint.")
    else:
        stopping_counter += 1
        if stopping_counter >= patience:
            print("Early stopping triggered.")
            break

print(f"Training finished. Best val acc: {best_val_acc:.4f}")

In [ ]:
# Test
import torch.nn.functional as F
from PIL import Image

def predict_image_full(img_path, model, device=DEVICE, patch_size=PATCH_SIZE, patch_crop=10, input_size=IMAGE_INPUT):
    """
    img_path: full-resolution image (3k x 4k)
    patch_crop: number of random crops (you can include deterministic center crop too)
    returns: avg_softmax, topk
    """
    model.eval()
    img = Image.open(img_path).convert("RGB")
    probs = []
    transform_patch = transforms.Compose([
        transforms.RandomResizedCrop(patch_size, scale=(0.9,1.0), ratio=(0.9,1.1)),
        transforms.Resize(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    # center crop once
    center = transforms.Compose([
        transforms.CenterCrop(patch_size),
        transforms.Resize(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    patches = []
    patches.append(center(img))
    for i in range(patch_crop-1):
        patches.append(transform_patch(img))
    xb = torch.stack(patches).to(device)
    with torch.no_grad():
        with autocast():
            logits = model(xb)
            soft = F.softmax(logits, dim=1)
    avg = soft.mean(dim=0)
    topk = torch.topk(avg, k=5)
    return avg.cpu().numpy(), [(int(idx), float(conf)) for idx, conf in zip(topk.indices, topk.values)]

# usage:
# avg_probs, topk = predict_image_full("/path/to/test.jpg", model)
# print(topk)
